### 📄 Cell documentation
Refer to [`analysis_first_cell.md`](analysis_first_cell.md) for explanation of the following code.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

data_path = "data/nodhavn.geojson"
nodhavn = gpd.read_file(data_path)

### 📄 Cell documentation
Refer to [`analysis_second_cell.md`](analysis_second_cell.md) for explanation of the following code.

In [ ]:
nodhavn.info()
print()
print(nodhavn.head())
print()
print("CRS:", nodhavn.crs)

### 📄 Cell documentation
Refer to [`analysis_third_cell.md`](analysis_third_cell.md) for explanation of the following code.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
nodhavn.plot(ax=ax, edgecolor="black")
ax.set_title("Nodhavn-området")
ax.set_xlabel("")
ax.set_ylabel("")
plt.axis("equal")
plt.show()

### 📄 Cell documentation
Refer to [`analysis_fourth_cell.md`](analysis_fourth_cell.md) for explanation of the following code.

In [ ]:
print("Columns:", list(nodhavn.columns))

# Try to find the category column (expected: "kategori")
category_col = None
for c in nodhavn.columns:
    if str(c).lower() in ["kategori", "category", "type"]:
        category_col = c
        break

fylke_col = next((c for c in nodhavn.columns if str(c).lower() == "fylke"), None)

if category_col is not None:
    print(f"\nUsing category column: {category_col}")
    print(
        "Unique kategori values:",
        sorted(nodhavn[category_col].dropna().unique().tolist()),
    )
    print("\nCounts by kategori:")
    print(nodhavn[category_col].value_counts(dropna=False).sort_index())
else:
    print("\nNo category-like column found (expected 'kategori').")

if fylke_col is not None:
    print(f"\nTop fylke by number of points (column: {fylke_col}):")
    print(nodhavn[fylke_col].value_counts().head(10))
else:
    print("\nNo 'fylke' column found.")

### 📄 Cell documentation
Refer to [`analysis_fifth_cell.md`](analysis_fifth_cell.md) for explanation of the following code.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

if category_col is None:
    print("No category column available; plotting points in a default color.")
    nodhavn.plot(ax=ax, color="steelblue", markersize=20)
else:
    # Convert to string so geopandas treats it as categorical/discrete values.
    nodhavn_to_plot = nodhavn.copy()
    nodhavn_to_plot[category_col] = nodhavn_to_plot[category_col].astype(str)

    nodhavn_to_plot.plot(
        ax=ax,
        column=category_col,
        categorical=True,
        legend=True,
        cmap="viridis",
        markersize=20,
        legend_kwds={"title": "Kategori"},
    )

ax.set_title("Nødhavn fordelt etter kategori (GeoJSON data)")
ax.set_axis_off()
plt.show()

### 📄 Cell documentation
Refer to [`analysis_sixth_cell.md`](analysis_sixth_cell.md) for explanation of the following code.

In [ ]:
# Project to a metric CRS for area computations (approximate in meters)
nodhavn_metric = nodhavn.to_crs(epsg=3857)

# Convex hull of all points: a simple way to describe the spatial spread
hull = nodhavn_metric.geometry.unary_union.convex_hull
area_km2 = hull.area / 1e6

centroid_metric = hull.centroid
centroid_lonlat = (
    gpd.GeoSeries([centroid_metric], crs=nodhavn_metric.crs)
    .to_crs(nodhavn.crs)
    .iloc[0]
)

print(f"Approx. convex hull area: {area_km2:.1f} km^2")
print(
    f"Approx. centroid (lon, lat): ({centroid_lonlat.x:.4f}, {centroid_lonlat.y:.4f})"
)

# Plot: points + convex hull outline + centroid
fig, ax = plt.subplots(figsize=(8, 8))
nodhavn_metric.plot(ax=ax, color="steelblue", markersize=10, alpha=0.6)

gpd.GeoSeries([hull], crs=nodhavn_metric.crs).boundary.plot(
    ax=ax, color="crimson", linewidth=2
)
gpd.GeoSeries([centroid_metric], crs=nodhavn_metric.crs).plot(
    ax=ax, color="black", markersize=30
)

ax.set_title("Geografisk utstrekning (convex hull) for nødhavnspunktene")
ax.set_axis_off()
plt.show()